# Study API実践：Ask-and-Tell・optimize・Active Learning・Mixed・q-batch

`Study` は、**候補を提案する `ask()`、実験結果を登録する `tell()`、一連の実験を自動実行する `optimize()`** を統一する公開APIです。

このNotebookでは、1点のAsk-and-Tellから始め、複数候補の状態管理、BOと回帰Active Learning、連続＋整数のMixed変数、候補間制約、DataFrameによる履歴確認へ進みます。独自の獲得関数や最適化器は実装しません。


## 1. 基本用語と実験の流れ

- **SearchSpace**：探索する変数の名前と上下限。例：`x`を0〜1の範囲で探索します。
- **Study**：試行（Trial）を管理し、次に試す候補を提案するオブジェクトです。
- **Trial**：候補パラメータ、試行番号、状態、観測値を持つ記録です。
- **ask**：候補を提案してTrialを`RUNNING`にします。
- **tell**：実験結果をStudyへ返し、Trialを`COMPLETE`または`FAIL`にします。

`WAITING → RUNNING → COMPLETE/FAIL`が基本の状態遷移です。最初の数回は初期候補生成を使うため、GPモデルの学習は不要です。

In [ ]:
import torch

from robotorchan.optim.constraints.contracts import CandidateConstraints
from robotorchan.optim.domains.variable_space import MixedVariableSpace
from robotorchan.study import TrialState, create_study

print("PyTorch:", torch.__version__)

### Trialの状態遷移と実験管理

`ask()` は候補を返すと同時にTrialを `RUNNING` にします。実験が成功したら `tell(trial, value)` で `COMPLETE`、失敗した場合は `FAIL` として扱います。

`ask_batch(q)` で提案された複数のTrialも、**各Trialの完了順序は自由**です。実験を並列に実行する機能そのものはStudy APIには含まれません。

## 2. Ask-and-Tell：1候補を評価する

ここでは最大化問題を考えます。真の目的関数は`f(x) = -(x - 0.7)^2`とし、最適値は`x = 0.7`で0です。実際の用途では、この計算部分を測定装置や実験担当者による評価に置き換えます。

`n_initial_trials=10`として、以下の少数回の試行では初期候補のみを使い、モデル学習の有無による挙動の違いを避けます。

In [ ]:
study = create_study(
    search_space={"x": (0.0, 1.0)},
    direction="maximize",
    n_initial_trials=10,
    seed=42,
)

trial = study.ask()
print("提案された候補:", trial.number, dict(trial.params), trial.state)
assert trial.state is TrialState.RUNNING

x = trial.params["x"]
observed_value = -(x - 0.7) ** 2
finished = study.tell(trial, observed_value)
print("観測結果:", finished.value, finished.state)
assert finished.state is TrialState.COMPLETE

### 最良観測値は何を表すか

`best_value` は**完了したTrialで実際に得られた目的関数値**です。GPの予測最大値や獲得関数値ではありません。

ここでは最大化問題なので、観測値が0に近いほど良い候補です。

## 3. q-batch：複数候補をまとめて提案する

`ask_batch(3)`は3個の候補を**同時に提案**します。戻り値は3つのTrialからなるタプルです。測定が終了した順に`tell()`を呼べるため、実験装置が複数ある場合にも使えます。

注意：**q-batchは候補をまとめて提案するAPI**です。実験を並列に実行する仕組みそのものは提供しません。外部の実験実行や並列処理は利用者が管理します。

In [ ]:
batch = study.ask_batch(3)
print("候補数:", len(batch))
for candidate in batch:
    print(candidate.number, dict(candidate.params), candidate.state)

for candidate in reversed(batch):
    value = -(candidate.params["x"] - 0.7) ** 2
    study.tell(candidate.number, value)

print("登録された試行数:", len(study.trials))
print("最良パラメータ:", dict(study.best_params))
print("最良目的関数値:", study.best_value)
assert len(study.trials) == 4

### まとめて提案しても測定の完了は別

`ask_batch(3)` で3候補を得ても、その時点では3件とも `RUNNING` です。下の例では逆順に `tell()` を呼び、実験終了順序が提案順序と一致する必要がないことを確認します。

## 4. `study.optimize()`：候補提案・評価・登録を自動化する

`ask()` と `tell()` を自分で繰り返す代わりに、`study.optimize(objective, n_trials=...)` は指定した回数だけ候補を提案・評価・登録します。

- `objective(trial)` は `trial.params` から入力を取り、観測した有限スカラー値を返します。
- `n_trials` は**今回追加する試行数**で、累計試行数ではありません。
- `best_params`、`best_value`、`best_trial` は完了済みの試行から得られます。

この例は `n_initial_trials=10`、実行4回なので**GPを使う前の初期候補生成**を示しています。モデルベース探索との違いは後半で説明します。


In [ ]:
def objective(trial):
    """候補xを受け取り、最大化したいスカラー評価値を返す。"""
    x = trial.params["x"]
    return -(x - 0.7) ** 2


optimize_study = create_study(
    search_space={"x": (0.0, 1.0)},
    direction="maximize",
    n_initial_trials=10,
    seed=123,
)
optimize_study.optimize(objective, n_trials=4)

print("試行数:", len(optimize_study.trials))
for result in optimize_study.trials:
    print(result.number, dict(result.params), result.value, result.state)

print("最良候補:", dict(optimize_study.best_params))
print("最良評価値:", optimize_study.best_value)
assert len(optimize_study.trials) == 4
assert all(result.state is TrialState.COMPLETE for result in optimize_study.trials)

### `optimize()` と手動Ask-and-Tellの使い分け

評価関数をPythonから直接呼び出せる場合は `optimize()` が便利です。一方、外部の装置や人による測定で完了まで待つ必要がある場合は、`ask()` で候補を受け取り、測定結果が届いてから `tell()` を呼ぶ方法が適しています。

### 4.1 途中経過を記録する：`callbacks`

`callbacks`には`callback(study, finished_trial)`という2引数の関数を渡します。各試行の終了後に呼ばれるので、ログ出力や外部の実験記録との連携に使えます。

In [ ]:
progress = []


def record_progress(study, finished_trial):
    progress.append((finished_trial.number, finished_trial.value))
    print(
        f"Trial {finished_trial.number}: "
        f"value={finished_trial.value:.4f}, best={study.best_value:.4f}"
    )


callback_study = create_study(
    search_space={"x": (0.0, 1.0)},
    n_initial_trials=10,
    seed=321,
)
callback_study.optimize(objective, n_trials=3, callbacks=[record_progress])
assert len(progress) == 3

### 4.2 評価失敗時の扱い：`catch`

`objective`で例外が発生すると、そのTrialは`FAIL`状態になります。既定では例外を再送出して処理を中断します。`catch=(ValueError,)`のように指定すると、対象の例外を記録して**次の試行へ継続**できます。

`catch`は指定した例外だけを捕捉します。想定外のエラーを隠さないよう、必要な例外型だけを指定してください。

In [ ]:
def sometimes_fails(trial):
    if trial.number == 1:
        raise ValueError("測定失敗を模擬")
    return objective(trial)


failure_study = create_study(
    search_space={"x": (0.0, 1.0)},
    n_initial_trials=10,
    seed=9,
)
failure_study.optimize(sometimes_fails, n_trials=3, catch=(ValueError,))
print([(t.number, t.state.value, t.value) for t in failure_study.trials])
assert [t.state for t in failure_study.trials] == [
    TrialState.COMPLETE,
    TrialState.FAIL,
    TrialState.COMPLETE,
]

### 失敗したTrialと成功したTrialを区別する

評価失敗を捕捉しても、失敗したTrialの値を正常な観測としてGPに渡すわけではありません。`FAIL` として履歴に残し、成功したTrialの `COMPLETE` と区別します。

## 5. Active Learning：予測の不確実性が大きい場所を学習する

ここまでの**ベイズ最適化（BO）**では、目的関数が大きくなる入力を探しました。一方、**Active Learning（AL）**では、最良の入力を探すことよりも、未知の関数を効率よく学習することを重視します。

同じ`Study.optimize()`でも、**獲得関数（acquisition）を切り替える**ことで探索の目的を変更できます。

| 観点 | 通常のBO | 回帰Active Learning |
| --- | --- | --- |
| 目標 | 目的関数の最大値を探す | 予測が不確かな領域の情報を集める |
| このNotebookでの獲得関数 | 既定の`qLogExpectedImprovement` | `qPosteriorVariance` |
| 候補の選択基準 | 既存の最良値に対する改善 | GPの予測分散 |

**注意**：ここで扱うのは**回帰のActive Learning**です。二値分類のBALDや予測エントロピーとは異なります。`objective`はALでも観測した応答値を返し、獲得関数のスコアを返すわけではありません。

以下は、両者に同じ観測関数を与え、獲得関数だけを変える最小例です。初期3点を使った後、4点目の候補生成でGPを学習します。

### 初期候補生成とモデルベース探索

`n_initial_trials=3` の場合、最初の3試行はGPを用いない初期候補です。**4試行目**から学習済みモデルと獲得関数に基づく候補提案に移ります。

BOでは改善を重視し、回帰ALでは予測分散を重視します。ALでも `tell()` に登録するのは**実際の応答値**であり、獲得関数のスコアではありません。

In [ ]:
from robotorchan.acquisition import qPosteriorVariance


def response_objective(trial):
    """実験で観測される応答値（最大化したいスコアとは限らない）。"""
    x = trial.params["x"]
    return -(x - 0.7) ** 2


def make_learning_study(*, acquisition=None):
    options = {
        "search_space": {"x": (0.0, 1.0)},
        "direction": "maximize",
        "n_initial_trials": 3,
        "seed": 24,
    }
    if acquisition is not None:
        options["acquisition"] = acquisition
    return create_study(**options)


bo_study = make_learning_study()
al_study = make_learning_study(acquisition=qPosteriorVariance)

# 最初の3回は両者とも初期候補生成を使用し、4回目から獲得関数で候補を選びます。
bo_study.optimize(response_objective, n_trials=4)
al_study.optimize(response_objective, n_trials=4)

print("BOの試行:", [(t.number, round(t.params["x"], 3)) for t in bo_study.trials])
print("ALの試行:", [(t.number, round(t.params["x"], 3)) for t in al_study.trials])
print("BOの最良値:", bo_study.best_value)
print("ALの観測中の最大値:", al_study.best_value)
assert len(bo_study.trials) == len(al_study.trials) == 4

### 5.1 結果の読み方と使い分け

`bo_study.best_value`はBOで最大化したい観測値です。一方、ALで得られた`best_value`は**観測値の最大値を機械的に表示しているだけ**で、ALの学習品質を評価する指標ではありません。

回帰ALの品質を評価するなら、未観測点での予測誤差、予測分散、学習曲線などを使います。また、`qPosteriorVariance`は候補点自身の分散に着目する基準であり、探索領域全体の不確実性低減を直接最適化するものではありません。領域全体を学習したい場合は、既存の`34_active_learning_acquisition.ipynb`にあるEPIGやBoTorchの`qNegIntegratedPosteriorVariance`も参照してください。

この比較では**4試行だけ**なので、候補が一致する場合や結果がばらつく場合があります。優劣のベンチマークではなく、Study APIで獲得関数を切り替える実装例として利用してください。

### BOとActive Learningの候補を比較する

BOとALは同じ初期条件から開始しても、4試行目以降に異なる点を提案する場合があります。以下では試行番号ごとの候補位置を表示します。

これは方式の違いを理解するための図であり、少数試行での性能比較や統計的な優劣判定ではありません。

In [ ]:
import matplotlib.pyplot as plt

bo_x = [t.params["x"] for t in bo_study.trials]
al_x = [t.params["x"] for t in al_study.trials]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(bo_x) + 1), bo_x, "o-", label="BO: improvement")
ax.plot(range(1, len(al_x) + 1), al_x, "s--", label="AL: posterior variance")
ax.axvline(3.5, linestyle=":", color="gray", label="After initial trials")
ax.set(xlabel="Trial number", ylabel="Proposed x", ylim=(-0.05, 1.05), title="BO vs Active Learning candidates")
ax.legend()
plt.show()

## 6. Mixed変数：連続変数と整数変数を組み合わせる

たとえば`x`を連続の配合比率、`count`を整数の処理回数とします。

`MixedVariableSpace`では、**探索変数の並び順**と`integer_dims`のインデックスを一致させます。下の例では`x`が0番、`count`が1番です。探索範囲の`bounds`は`[2, d]`形状で、上段が下限、下段が上限です。

`CandidateConstraints`の不等式は、この例では`count >= 3`を意味します。

### 連続変数と整数変数の区別

`x` は連続値、`count` は整数です。`MixedVariableSpace` の `integer_dims=(1,)` は、**2番目の変数が整数**であることを意味します。

線形不等式 `count >= 3` は既知の候補制約です。整数性と候補制約の両方を満たす入力だけが提案されます。

In [ ]:
variable_space = MixedVariableSpace(
    bounds=torch.tensor([[0.0, 1.0], [1.0, 4.0]], dtype=torch.double),
    integer_dims=(1,),
)
constraints = CandidateConstraints(
    inequality_constraints=((torch.tensor([1]), torch.tensor([1.0]), 3.0),),
)
mixed_study = create_study(
    search_space={"x": (0.0, 1.0), "count": (1.0, 4.0)},
    variable_space=variable_space,
    constraints=constraints,
    n_initial_trials=10,
    seed=7,
)
mixed_trials = mixed_study.ask_batch(3)
for candidate in mixed_trials:
    print("提案:", candidate.number, dict(candidate.params))
    assert candidate.params["count"] in (3.0, 4.0)
    mixed_study.tell(candidate, candidate.params["x"])

print("Mixed実験の最良候補:", dict(mixed_study.best_params))

### Mixed候補を図で確認する

整数変数 `count` は連続的に変化するのではなく、許された整数値に限られます。提案された候補の `x` と `count` を散布図で確認します。

In [ ]:
mixed_x = [t.params["x"] for t in mixed_trials]
mixed_count = [t.params["count"] for t in mixed_trials]
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(mixed_x, mixed_count, s=90)
ax.set(xlabel="Continuous x", ylabel="Integer count", xlim=(0, 1), ylim=(0.5, 4.5), yticks=[1, 2, 3, 4], title="Mixed candidates under count >= 3")
plt.show()

## 7. 外部から渡す候補も検証される

実験済み候補や利用者が指定した候補は`add_trial()`で`WAITING`状態として登録できます。Mixed変数では整数性を、制約付きStudyでは候補の可行性を検証します。以下は**例外が発生することが正しい動作**です。

In [ ]:
try:
    mixed_study.add_trial({"x": 0.5, "count": 2.5})
except ValueError as exc:
    print("整数条件に違反:", exc)

try:
    mixed_study.add_trial({"x": 0.5, "count": 2.0})
except ValueError as exc:
    print("制約に違反:", exc)

waiting = mixed_study.add_trial({"x": 0.5, "count": 3.0})
running = mixed_study.start_trial(waiting.number)
mixed_study.tell(running, 0.5)
print("外部候補の登録完了:", mixed_study.get_trial(waiting.number).state)

### 外部から登録する候補も同じ制約に従う

`add_trial()` は既存の実験計画などから候補を取り込むためのAPIです。ただし、整数変数に小数を入れたり、既知制約に違反する入力を渡したりすると拒否されます。

外部から与えた候補も `WAITING` → `RUNNING` → `COMPLETE` の状態遷移を経て登録します。

## 8. 候補間制約：q個の候補をまとめて評価する

単一候補の制約`x >= 0.8`と、**2候補の合計**に関する制約`x₀ + x₁ >= 1.5`は異なります。

下の`indices`は`[[候補番号, 変数番号], ...]`の形状です。`[[0, 0], [1, 0]]`は、0番目と1番目の候補の`x`を参照します。係数`[1, 1]`と右辺`1.5`で`x₀ + x₁ >= 1.5`になります。

候補間制約があるときは、`ask()`で1点ずつ提案するのではなく、制約で指定した候補数に対応する`ask_batch(q)`を使用します。

### 候補間制約は各候補を独立に評価できない

`x0+x1 >= 1.5` は2候補の**組全体**に課す条件です。各候補に対する `x >= 0.8` とは異なり、1点ずつ独立に `ask()` すると制約の意味を保てません。

`ask_batch(2)` を使い、提案された2点の和が条件を満たすことを確認します。

In [ ]:
joint_constraints = CandidateConstraints(
    inequality_constraints=(
        (
            torch.tensor([[0, 0], [1, 0]]),
            torch.tensor([1.0, 1.0]),
            1.5,
        ),
    ),
)
joint_study = create_study(
    search_space={"x": (0.0, 1.0)},
    constraints=joint_constraints,
    n_initial_trials=10,
    seed=3,
)
joint_trials = joint_study.ask_batch(2)
values = [candidate.params["x"] for candidate in joint_trials]
print("候補:", values, "合計:", sum(values))
assert sum(values) >= 1.5 - 1e-7

for candidate in joint_trials:
    joint_study.tell(candidate, -(candidate.params["x"] - 0.7) ** 2)

### q-batchの制約を図で確認する

候補2点の和が1.5以上であることを、2次元平面上の可行領域として示します。図の各軸は異なる候補の `x` 値であり、**1候補が2次元入力を持つという意味ではありません**。

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0.5, 1.0], [1.0, 0.5], linestyle="--", label="x0 + x1 = 1.5")
ax.scatter(values[0], values[1], marker="*", s=180, label="Proposed joint batch")
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="Candidate 0: x", ylabel="Candidate 1: x", title="Inter-point feasible batch")
ax.legend()
plt.show()

## 9. 実験履歴をDataFrameで確認する

`trials_dataframe()`は`number`（試行番号）、`state`（状態）、`value`（目的関数値）、探索変数列を持つ表を返します。`states=(TrialState.COMPLETE,)`で完了済みの試行に絞れます。

pandasをインストールしていない場合は、`pip install "robotorchan[dataframe]"`で追加できます。このセルでは未インストール時に案内を表示します。

In [ ]:
try:
    import pandas as pd
except ImportError:
    print('DataFrame表示には pip install "robotorchan[dataframe]" が必要です。')
else:
    display(study.trials_dataframe())
    display(study.trials_dataframe(states=(TrialState.COMPLETE,)))

### DataFrameでTrial履歴を確認する

DataFrameには試行番号、状態、目的関数値、パラメータが含まれます。`COMPLETE` のみ抽出すれば、失敗した試行を除いて観測履歴を確認できます。

pandasは任意依存なので、未インストールでもStudy API自体は利用できます。

## 10. まとめと実利用時の注意

- `ask()` / `ask_batch(q)`で候補を受け取り、外部実験後に`tell()`で観測値を返します。
- `Trial`は状態遷移を持つため、同じ試行を二重に完了させることはできません。
- Mixed変数では変数の順序と`integer_dims`を一致させます。
- 候補間制約は個々の候補ではなく、**q-batch全体**に適用されます。
- 制約付き初期候補は最大**2048回**の棄却サンプリングで生成します。可行領域が狭い場合や可行解が存在しない場合は失敗します。
- `Study.optimize()`は現時点では逐次実行です。モデル学習後のMixed / q-batch探索では、選択したモデル・獲得関数・Optimizerの互換性を別途確認してください。

**次のステップ**：目的関数を実際の測定値に置き換え、`ask()`と`tell()`を外部の実験ワークフローへ組み込んでみてください。

## 11. APIの役割と次のステップ

| 目的 | 公開API | 注意点 |
|---|---|---|
| Study作成 | `create_study()` | 方向・探索空間・初期試行数 |
| 1候補の提案 | `ask()` | TrialはRUNNING |
| q候補の提案 | `ask_batch(q)` | 並列実験の実行機構ではない |
| 観測値の登録 | `tell()` | 完了した実験結果を渡す |
| 自動評価ループ | `study.optimize()` | objectiveは実測値を返す |
| BOと回帰ALの切替 | `acquisition=qPosteriorVariance` など | ALのスコアを観測値として登録しない |
| Mixed変数 | `MixedVariableSpace` | 整数次元を明示 |
| 既知の制約 | `CandidateConstraints` | 候補間制約はjoint qで判定 |
| 履歴出力 | `trials_dataframe()` | pandasは任意依存 |

**次の実務ステップ：** 人や装置による実験を `ask()` と `tell()` に接続し、必要に応じてモデル・獲得関数・Optimizerの互換性を確認してください。